<a href="https://colab.research.google.com/github/TaherBenAfia/Fly2/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)

(30000, 44)


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [2]:
print(len(df), df['content_id'].is_unique, df['client_id'].nunique())
print(df['content_age_days'].min(), df['content_age_days'].max())
print(df['days_since_last_update'].min(), df['days_since_last_update'].max())

30000 True 32
90 564
1 373


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [3]:
label = ['trend_direction']
features = ['search_volume','competition','cpc','word_count','char_count','content_age_days',
            'days_since_last_update','ctr','avg_position','engagement_rate','scroll_rate','ai_traffic_pct',
            'competition_level','content_type','main_intent','age_tier','freshness_tier',
            'word_count_tier','impression_tier','position_tier']
context = ['content_id','client_id']
excluded = ['trend_pct','provider_used','model_used']
print(label)
print(features)
print(context)
print(excluded)

['trend_direction']
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']
['content_id', 'client_id']
['trend_pct', 'provider_used', 'model_used']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
import numpy as np

print(len(df), df['content_id'].is_unique, df['client_id'].nunique())
print(df['content_age_days'].min(), df['content_age_days'].max())
print(df['days_since_last_update'].min(), df['days_since_last_update'].max())

miss = df.isna().sum()
print(miss[miss > 0].sort_values(ascending=False))

kw_cols = ['search_volume', 'competition', 'cpc']
print(df.groupby('content_type')[kw_cols].apply(lambda g: g.isna().mean()))

print((df['avg_position'] == 0).sum())

print(df['trend_direction'].value_counts())
print(df['trend_direction'].eq('down').mean().round(3))

sub = df[['impressions_90d', 'impressions_last_30d', 'impressions_prev_30d']].dropna()
ratio = (sub['impressions_last_30d'] + sub['impressions_prev_30d']) / sub['impressions_90d'].replace(0, np.nan)
print(ratio.mean().round(3))

30000 True 32
90 564
1 373
provider_used        21438
word_count_tier       7699
char_count            7699
word_count            7699
char_count_tier       7699
model_used            5733
trend_pct             3388
competition_level     2610
search_volume         2468
competition           2468
cpc                   2468
main_intent           2374
scroll_rate            125
dtype: int64
                    search_volume  competition       cpc
content_type                                            
comparison article       0.000000     0.000000  0.000000
feedly article           1.000000     1.000000  1.000000
keyword article          0.013673     0.013673  0.013673
1205
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64
0.542
0.562


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [5]:
print('seasonality: no calendar field survived anonymization, age/trend confound untestable')
print('window overlap: impressions_90d overlaps the labels 30d windows by', ratio.mean().round(3))
print('missingness by content_type is systematic, not random, confirmed above')
print('single 90d snapshot: no forward-looking validity, cross-sectional only')

seasonality: no calendar field survived anonymization, age/trend confound untestable
window overlap: impressions_90d overlaps the labels 30d windows by 0.562
missingness by content_type is systematic, not random, confirmed above
single 90d snapshot: no forward-looking validity, cross-sectional only


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.